# 05 — Question 3: changes in exposure and changes in wages (within-person)

**Question:** Among workers observed in several waves, are changes in occupational AI exposure associated with changes in wages, once individual fixed characteristics, year effects and changes in occupational skill level are controlled for? How does this within-person association compare with the cross-sectional one?

**Rule:** no individual rows are displayed, only coefficients and counts.

**Idea.** Question 2 compares **different** individuals. If exposed occupations pay more, it may be because more qualified people work in them.
Here, we compare **each individual with themselves**: everything that is fixed about a person (ability, background, quality of their degree) drops out.
The association is then identified **only by individuals who change occupation**.

| Model | Comparison | What is removed |
|---|---|---|
| 1. Pooled OLS | across individuals (as in Q2) | nothing |
| 2. Individual fixed effects | each individual with themselves | everything that is fixed about the individual |
| 3. Fixed effects + major ISCO group | same, at constant skill level | the effect of promotions / changes in skill level |

The three models are estimated on **tertiary graduates** (main specification, consistent with the research question), then on **all workers** (robustness).

**Course methods:** `groupby().transform()` (Week 2), `get_dummies` (Week 2), `LinearRegression` (Week 6).
**Beyond the course (flagged):** individual fixed effects, and `statsmodels` for standard errors clustered by individual.

In [ ]:
# If statsmodels is missing: run  %pip install statsmodels  once in a cell, then Restart
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression

DATA_PATH = "../Data/private/shp_isco4_ilo.parquet"

## 1. Prepare the sample
- We keep only **precise codes** (`exposure_source == "isco4"`): vague codes (`2000`, `2500`…) create spurious occupation changes.
- We keep individuals observed **at least 2 years**: with a single observation, no comparison with oneself is possible.
- `expo10` = exposure × 10: the coefficient then reads as the effect of a **0.1** increase in exposure, the order of magnitude of a typical occupation change (observed median: 0.09).

In [ ]:
df = pd.read_parquet(DATA_PATH)

def prepare(d):
    d = d[(d["exposure_source"] == "isco4") & (d["sex"].isin([1, 2]))].copy()
    n_years = d.groupby("idpers")["year"].transform("nunique")
    d = d[n_years >= 2].copy()

    d["expo10"] = d["exposure_main"] * 10
    d["age2"] = d["age"] ** 2
    d["isco1"] = d["isco4"].str[:1]
    d["isco2"] = d["isco4"].str[:2]
    for col in ["sex", "isced", "region", "year"]:
        d[col] = d[col].astype("Int64").astype(str)
    return d

samples = {
    "tertiary": prepare(df[df["tertiary"] == 1]),
    "all": prepare(df),
}

for name, d in samples.items():
    chg_isco2 = (d.groupby("idpers")["isco2"].nunique() > 1).sum()
    chg_expo = (d.groupby("idpers")["expo10"].nunique() > 1).sum()
    print(f"{name:>8}: {len(d)} rows | {d['idpers'].nunique()} individuals | "
          f"{chg_isco2} change 2-digit ISCO | {chg_expo} have a change in exposure")

## 2. Fixed effects: subtract each individual's mean
The fixed effect $\alpha_i$ captures everything that is constant for individual $i$. In the model
$$\log w_{it} = \beta\,\text{expo}_{it} + \gamma_t + \alpha_i + X_{it}'\delta + \varepsilon_{it},$$
we subtract from each variable **the individual's own mean**. $\alpha_i$ disappears, since a constant minus its mean equals 0.
What remains are **each individual's deviations from their own mean**. This is the *within* estimator.

Consequences:
- time-invariant variables (sex, education level) also disappear: they can no longer be estimated;
- **linear age is dropped**: for a given individual, age increases exactly like the year, so it is collinear with the year dummies. Age squared is kept.

In [ ]:
def demean(X, groups):
    return X - X.groupby(groups).transform("mean")

## 3. Estimate the three models
Standard errors **clustered by individual**: observations of the same individual are not independent.
`coef_%` = wage change in % for a 0.1 increase in exposure (log approximation).

In [ ]:
def estimate(d, sample_name):
    y = d["log_income"]
    years = pd.get_dummies(d["year"], prefix="y", drop_first=True, dtype=int)
    fixed_chars = pd.get_dummies(d[["sex", "isced", "region"]], drop_first=True, dtype=int)
    major_group = pd.get_dummies(d["isco1"], prefix="isco1", drop_first=True, dtype=int)
    clusters = d["idpers"]

    specs = {
        "1_pooled_OLS": (pd.concat([d[["expo10", "age", "age2"]], years, fixed_chars], axis=1), False),
        "2_fixed_effects": (pd.concat([d[["expo10", "age2"]], years], axis=1), True),
        "3_FE_+_major_group": (pd.concat([d[["expo10", "age2"]], years, major_group], axis=1), True),
    }

    rows = []
    for name, (X, fixed_effects) in specs.items():
        if fixed_effects:
            X_est, y_est = demean(X, clusters), demean(y, clusters)
        else:
            X_est, y_est = sm.add_constant(X), y
        res = sm.OLS(y_est, X_est).fit(cov_type="cluster", cov_kwds={"groups": clusters})
        b, se = res.params["expo10"], res.bse["expo10"]
        rows.append({"sample": sample_name, "model": name,
                     "coef_%": 100 * b, "CI95_low_%": 100 * (b - 1.96 * se),
                     "CI95_high_%": 100 * (b + 1.96 * se), "p_value": res.pvalues["expo10"],
                     "n_rows": int(res.nobs), "n_individuals": clusters.nunique()})
    return rows

results = []
for name, d in samples.items():
    results += estimate(d, name)
results = pd.DataFrame(results).set_index(["sample", "model"])
results.round(3)

**Link with the course:** the fixed-effects coefficient is exactly that of a `LinearRegression` (Week 6) on the demeaned data. `statsmodels` is only used to obtain clustered standard errors.

In [ ]:
d = samples["tertiary"]
years = pd.get_dummies(d["year"], prefix="y", drop_first=True, dtype=int)
X_dm = demean(pd.concat([d[["expo10", "age2"]], years], axis=1), d["idpers"])
y_dm = demean(d["log_income"], d["idpers"])

lr = LinearRegression(fit_intercept=False).fit(X_dm, y_dm)
print("expo10 coefficient (LinearRegression):", round(100 * lr.coef_[0], 3), "%")
print("expo10 coefficient (statsmodels):     ", round(results.loc[("tertiary", "2_fixed_effects"), "coef_%"], 3), "%")

## 4. How to read the results

| Comparison | If… | Then… |
|---|---|---|
| **1 → 2** | the coefficient drops sharply | the cross-sectional link (Q2) mostly reflects **who** works in exposed occupations (sorting) |
| | the coefficient stays similar | the link is not explained by fixed characteristics of individuals |
| **2 → 3** | the coefficient disappears | the link runs through the **skill level** of the occupation (changes in major group, promotions) |
| | the coefficient remains | at constant skill level, moving to a more exposed occupation is still associated with wages |
| **tertiary vs all** | same direction | the result is robust to the sample |

A confidence interval that contains 0 means the association cannot be distinguished from zero.

## 5. Conclusion

| Sample | Model | Coef. (% per +0.1 exposure) | 95% CI | p |
|---|---|---|---|---|
| Tertiary | 1. Pooled OLS | +2.6 | [1.8; 3.3] | < 0.001 |
| | 2. Individual FE | +2.0 | [−0.1; 4.2] | 0.061 |
| | 3. FE + major group | +2.5 | [−0.1; 5.1] | 0.055 |
| All workers | 1. Pooled OLS | +3.7 | [3.2; 4.2] | < 0.001 |
| | 2. Individual FE | +1.2 | [−0.0; 2.5] | 0.058 |
| | 3. FE + major group | +2.0 | [0.1; 3.9] | 0.035 |

Identification comes from 692 tertiary graduates (1,311 workers overall) whose occupational exposure changes over the period.

**Answer to Q3.** Changes in occupational exposure are positively but weakly associated with wage changes within individuals.

- **Sorting.** Among all workers, the within-person association (+1.2% per 0.1 increase in exposure) is about one third of the cross-sectional one (+3.7%). Most of the cross-sectional link therefore reflects which workers sort into exposed occupations, rather than the occupation itself. Among tertiary graduates, sorting matters less (+2.6% pooled vs +2.0% within), as expected in a sample with more homogeneous education.
- **Skill level.** Controlling for changes in major occupational group (1-digit ISCO) raises the within estimate (+2.0% among all workers, 95% CI [0.1; 3.9]). A plausible explanation is that moves into clerical occupations (major group 4), which are the most exposed to generative AI but relatively low-paid, mask a positive association at constant skill level. Intuitively, when a worker moves down from professional to clerical work, exposure rises while wages fall because of the change in skill level, not because of exposure itself; controlling for major group attributes this wage drop to the group change rather than to exposure. This interpretation is a hypothesis and was not tested directly.
- **Link with Q2.** The two results are consistent. A wage difference of about 2% per 0.1 of exposure is statistically detectable within individuals, but too small to reduce prediction error noticeably, given a typical prediction error of about 36% (Q2).

**Limitations.** (1) Within-person estimates are imprecise: only one coefficient is significant at the 5% level, so results should be read as suggestive. (2) Part of the remaining occupation changes may be coding noise, which biases fixed-effects estimates towards zero. (3) Fixed effects remove time-invariant characteristics, but not events that change together with the occupation (promotions, change of employer), so the estimates are associations, not causal effects. (4) Occupation changers are not a random subset of workers. (5) The exposure index is fixed (ILO 2025): exposure changes only when the occupation changes, so these results describe wage differences across occupations, not the arrival of AI over time.